In [15]:
%pip install google-genai

Note: you may need to restart the kernel to use updated packages.


In [33]:
import getpass
from google import genai

api_key = getpass.getpass("Pegá tu API key: ")
client  = genai.Client(api_key=api_key)
print("Cliente de Gemini listo")

Cliente de Gemini listo


In [34]:
for modelo in client.models.list():
    print(modelo.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-omni-1.2-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/lyria-3-clip-preview
models

In [35]:
def prioridad_modelo(nombre):
    puntaje= 100
    if "flash" in nombre: puntaje -=50
    if "pro" in nombre: puntaje -=20
    if "preview" in nombre or "latest" in nombre: puntaje +=30
    return puntaje


In [36]:
import time

todos_los_modelos=[m.name for m in client.models.list() ]
candidatos = [m for m in todos_los_modelos if "gemini" in m and "tts" not in m and "image" not in m and "embedding" not in m and "embedding" not in m and "live" not in m and "robotics" not in m]
candidatos = sorted(candidatos,key=prioridad_modelo)

MODELO = None
for modelo in candidatos:
    nombre_limpio = modelo.replace("models/", "")
    print(f"-> Probando {nombre_limpio} ...", end=" ")
    try:
        respuesta = client.models.generate_content(model=nombre_limpio, contents="Respondé solo con la palabra: funciona")
        MODELO = nombre_limpio
        print("ÉXITO:", respuesta.text)
        break
    except Exception as e:
        print(f"FALLÓ ({type(e).__name__})")
        time.sleep(1)

print("Modelo elegido:", MODELO)

-> Probando gemini-2.5-flash ... FALLÓ (ClientError)
-> Probando gemini-2.5-flash-lite ... FALLÓ (ClientError)
-> Probando gemini-3.1-flash-lite ... ÉXITO: funciona
Modelo elegido: gemini-3.1-flash-lite


In [37]:
import json
import time
from PIL import Image
from google.genai import types
import pandas as pd

tabla = pd.read_csv("afiches.csv")

prompt_caracteristicas = """
1. Rol del agente

Sos un agente de inteligencia artificial especializado en análisis visual y descripción de imágenes aplicada a la comunicación política.

Tu tarea consiste en analizar afiches políticos y registrar sus características para construir después variables cuantificables.

En esta etapa analizá solo el afiche que te envío.

2. Análisis y extracción de características

Identificá y extraé todas las características visibles que consideres relevantes: textos tal como aparecen, colores, tipografía, imágenes, personas y símbolos.

Registrá solo lo que se ve. No agregues interpretaciones sobre su significado, tono, destinatario ni identidad política.

Ignorá los logos de la UBA y del Observatorio de Campañas Electorales: son marca de agua del archivo.

4. Almacenamiento de las características en JSON

Devolvé un JSON con esta estructura:

{
  "id_afiche": "ID del afiche que te indico",
  "caracteristicas": {}
}

En "caracteristicas" registrá todo lo que encuentres relevante en el afiche. Elegí vos los nombres de las claves, cómo organizarlas y qué valores darles.
"""

caracteristicas = []
for fila in tabla.itertuples():
    imagen = Image.open(fila.archivo)
    respuesta = client.models.generate_content(
        model=MODELO,
        contents=[imagen, prompt_caracteristicas, f"El ID de este afiche es {fila.id_afiche}."],
        config=types.GenerateContentConfig(response_mime_type="application/json"),
    )
    caracteristicas.append(json.loads(respuesta.text))
    time.sleep(5)

with open("caracteristicas_afiches.json", "w", encoding="utf-8") as f:
    json.dump(caracteristicas, f, ensure_ascii=False, indent=2)

KeyboardInterrupt: 

In [ ]:
print(len(caracteristicas))
print(respuesta.text[-300:])

59
      "blanco"
      ],
      "fondo": "blanco con marcas de agua tenues",
      "disposicion": "horizontal con texto sobre banda azul en la parte inferior"
    },
    "tipografia": {
      "nombre": "CRISTINA": "sans-serif, mayúsculas, negrita",
      "eslogan": "sans-serif, mayúsculas"
    }
  }
}


In [ ]:
hechos = [c["id_afiche"] for c in caracteristicas]
fallidos = []

for fila in tabla.itertuples():
    if fila.id_afiche in hechos:
        continue
    imagen = Image.open(fila.archivo)
    respuesta = client.models.generate_content(
        model=MODELO,
        contents=[imagen, prompt_caracteristicas, f"El ID de este afiche es {fila.id_afiche}."],
        config=types.GenerateContentConfig(response_mime_type="application/json"),
    )
    try:
        caracteristicas.append(json.loads(respuesta.text))
    except json.JSONDecodeError:
        fallidos.append(fila.id_afiche)
    time.sleep(5)

with open("caracteristicas_afiches.json", "w", encoding="utf-8") as f:
    json.dump(caracteristicas, f, ensure_ascii=False, indent=2)
print("Guardados:", len(caracteristicas), "| Fallidos:", fallidos)

Guardados: 141 | Fallidos: []


In [ ]:
with open("caracteristicas_afiches.json", encoding="utf-8") as f:
    caracteristicas = json.load(f)

prompt_definir = """
Sos un agente especializado en análisis estadístico aplicado a la comunicación política.
Te paso un JSON con las características observadas en afiches de campañas presidenciales argentinas.

1. Construí entre 5 y 10 variables independientes que te parezcan relevantes para determinar si un afiche es peronista o no, a partir de las características que se repiten en los JSON. Pueden ser dicotómicas, categóricas, ordinales o cuantitativas.
2. No construyas variables que identifiquen directamente al partido, al candidato o a la campaña: nombres de partidos o de dirigentes, palabras como "peronista" o "justicialista", siglas o escudos partidarios.
3. Definí la variable dependiente Peronista: 1 si el afiche es peronista y 0 si no lo es.
4. No infieras la orientación política exclusivamente por un color, símbolo o elemento aislado.

Devolvé un JSON con dos claves:
- "definiciones": cada variable con su criterio de codificación.
- "formula": la fórmula de la regresión, por ejemplo "Peronista ~ variable_1 + variable_2".
"""

respuesta = client.models.generate_content(
    model=MODELO,
    contents=[prompt_definir, json.dumps(caracteristicas, ensure_ascii=False)],
    config=types.GenerateContentConfig(response_mime_type="application/json"),
)
resultado = json.loads(respuesta.text)
print(resultado["definiciones"])
print(resultado["formula"])

{'v_1_presencia_identidad_nacional': "Dicotómica (1/0): El afiche incluye símbolos patrios, mapas del país, el Sol de Mayo o menciones explícitas a la 'Nación' o 'Patria' vinculadas directamente a la gestión del candidato.", 'v_2_uso_terminologia_obrera': "Dicotómica (1/0): Presencia de palabras como 'trabajadores', 'obreros', 'descamisados', 'fuerza productiva' o 'sindicatos'.", 'v_3_referencia_a_la_victoria': "Dicotómica (1/0): Uso de la palabra 'victoria' en el eslogan o nombre de la alianza (excluyendo nombres propios de partidos).", 'v_4_paleta_cromatica_enfasis_naranja': 'Dicotómica (1/0): El afiche utiliza el naranja como color predominante o acento principal en logotipos, eslóganes o elementos decorativos.', 'v_5_frecuencia_de_retratos': 'Cuantitativa: Cantidad de personas retratadas o mencionadas en el afiche (número entero).', 'v_6_estilo_fotografico_monocromatico': 'Dicotómica (1/0): El afiche utiliza predominantemente blanco y negro o tonos sepia (típico de épocas previas a

In [43]:
import os

with open("caracteristicas_afiches.json", encoding="utf-8") as f:
    caracteristicas = json.load(f)

prompt_codificar = (
    "Usando estas definiciones de variables, codificá el afiche descripto en el JSON. "
    "Devolvé un JSON con \"ID\", \"Peronista\" y el valor de cada variable de las definiciones, "
    "con exactamente los mismos nombres.\n\n"
    + json.dumps(resultado, ensure_ascii=False)
)

datos = []
if os.path.exists("variables_afiches.json"):
    with open("variables_afiches.json", encoding="utf-8") as f:
        datos = json.load(f).get("datos", [])
        datos = [x[0] if isinstance(x, list) else x for x in datos]


hechos = [x["ID"] for x in datos]
fallidos = []

for c in caracteristicas:
    if c["id_afiche"] in hechos:
        continue
    try:
        r = client.models.generate_content(
            model=MODELO,
            contents=[prompt_codificar, json.dumps(c, ensure_ascii=False)],
            config=types.GenerateContentConfig(response_mime_type="application/json"),
        )
        fila = json.loads(r.text)
        if isinstance(fila, list):
            fila = fila[0]
        datos.append(fila)
    except Exception as e:
        fallidos.append(c["id_afiche"])
    resultado["datos"] = datos
    with open("variables_afiches.json", "w", encoding="utf-8") as f:
        json.dump(resultado, f, ensure_ascii=False, indent=2)
    time.sleep(5)

print("Codificados:", len(datos), "| Fallidos:", fallidos)

Codificados: 141 | Fallidos: []


In [44]:
%pip install statsmodels

import numpy as np
import statsmodels.formula.api as smf

df = pd.DataFrame(resultado["datos"])
variables = df.columns.drop("ID")
df[variables] = df[variables].apply(pd.to_numeric)
d = df
m2 = smf.logit(resultado["formula"], data=d).fit()
print(m2.summary())
print(np.exp(m2.params))
print(np.exp(m2.conf_int()))

Note: you may need to restart the kernel to use updated packages.
Optimization terminated successfully.
         Current function value: 0.577269
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:              Peronista   No. Observations:                  141
Model:                          Logit   Df Residuals:                      135
Method:                           MLE   Df Model:                            5
Date:                Sat, 07 Nov 2026   Pseudo R-squ.:                  0.1647
Time:                        16:42:20   Log-Likelihood:                -81.395
converged:                       True   LL-Null:                       -97.446
Covariance Type:            nonrobust   LLR p-value:                 5.669e-06
                                       coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------

In [47]:
prompt_definir = """
Sos un agente especializado en análisis estadístico aplicado a la comunicación política.
Te paso un JSON con las características observadas en un conjunto de afiches políticos.

1. Construí entre 5 y 10 variables independientes que te parezcan relevantes para determinar si un afiche es peronista o no, a partir de las características que se repiten en los JSON. Pueden ser dicotómicas, categóricas, ordinales o cuantitativas.
2. No construyas variables que identifiquen al emisor: nombres de partidos, alianzas, listas o dirigentes, ni palabras o consignas que los contengan.
3. Cada característica observada debe pertenecer a una sola variable: que dos variables no midan lo mismo.
4. Elegí características que aparezcan en una cantidad suficiente de afiches de los dos grupos.
5. Definí la variable dependiente Peronista: 1 si el afiche es peronista y 0 si no lo es.
6. No infieras la orientación política exclusivamente por un color, símbolo o elemento aislado.

Devolvé un JSON con dos claves:
- "definiciones": cada variable con su criterio de codificación.
- "formula": la fórmula de la regresión, por ejemplo "Peronista ~ variable_1 + variable_2".
"""

In [49]:
import os

if os.path.exists("variables_afiches.json"):
    os.rename("variables_afiches.json", "variables_ronda1.json")

with open("variables_ronda1.json", encoding="utf-8") as f:
    anterior = json.load(f)

prompt_ronda2 = (
    prompt_definir
    + "\nEn la ronda anterior propusiste estas variables:\n"
    + json.dumps(anterior["definiciones"], ensure_ascii=False)
    + "\n\nEstos son los resultados de la regresión logística:\n"
    + m2.summary().as_text()
    + "\n\nProponé un nuevo conjunto de variables que mejore el modelo. "
    "Conservá las que tuvieron peso y reemplazá las que no."
)

respuesta = client.models.generate_content(
    model=MODELO,
    contents=[prompt_ronda2, json.dumps(caracteristicas, ensure_ascii=False)],
    config=types.GenerateContentConfig(response_mime_type="application/json"),
)
resultado = json.loads(respuesta.text)
print(resultado["definiciones"])
print(resultado["formula"])

{'v_1_presencia_identidad_nacional': 'Dicotómica (1/0): Incluye símbolos patrios (Sol de Mayo, banderas, mapa de Argentina) vinculados a la gestión o candidato.', 'v_2_uso_terminologia_obrera': "Dicotómica (1/0): Presencia de términos como 'trabajadores', 'obreros', 'descamisados', 'justicia social' o 'fuerza productiva'.", 'v_3_referencia_a_la_victoria': "Dicotómica (1/0): El eslogan contiene la palabra 'victoria' o derivaciones vinculadas a la coalición (excluyendo nombres propios de partidos).", 'v_4_paleta_cromatica_enfasis_naranja': 'Dicotómica (1/0): El naranja es utilizado como color predominante o acento principal en logotipos o elementos decorativos.', 'v_5_frecuencia_de_retratos': 'Cuantitativa: Cantidad total de personas retratadas o mencionadas explícitamente en el afiche.', 'v_6_uso_de_eslogan_interpelacion_directa': "Dicotómica (1/0): El texto incluye apelaciones directas al espectador (ej. 'vos', 'invitarme', 'tu voto', 'saben cómo hacer').", 'v_7_presencia_gesto_victori

In [50]:
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

m2_ronda1 = m2
d_ronda1 = d.copy()

d_ronda1["campana"] = d_ronda1["ID"].str[:3]
campanas_prueba = d_ronda1["campana"].drop_duplicates().sample(frac=0.25, random_state=42)
entrena = d_ronda1[~d_ronda1["campana"].isin(campanas_prueba)]
prueba = d_ronda1[d_ronda1["campana"].isin(campanas_prueba)]

m3 = smf.logit(m2_ronda1.model.formula, data=entrena).fit()
prob = m3.predict(prueba)
pred = (prob >= 0.5).astype(int)

print("Entrenamiento:", len(entrena), "| Prueba:", len(prueba))
print(confusion_matrix(prueba["Peronista"], pred))
print(classification_report(prueba["Peronista"], pred))
print("AUC:", roc_auc_score(prueba["Peronista"], prob))
print("Baseline:", (prueba["Peronista"] == entrena["Peronista"].mode()[0]).mean())

Optimization terminated successfully.
         Current function value: 0.555814
         Iterations 5
Entrenamiento: 103 | Prueba: 38
[[ 8  0]
 [21  9]]
              precision    recall  f1-score   support

           0       0.28      1.00      0.43         8
           1       1.00      0.30      0.46        30

    accuracy                           0.45        38
   macro avg       0.64      0.65      0.45        38
weighted avg       0.85      0.45      0.46        38

AUC: 0.7770833333333333
Baseline: 0.21052631578947367
